In [4]:
# Imports

import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [9]:
# Configurar scanpy para mejores gráficos y más info

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor="white")

In [2]:
# Definir rutas para no perderme

DATA_DIR = "../data"
RESULTS_DIR = "../results"
FIGURES_DIR = "../figures"

In [11]:
# Ver datasets que tengo

datasets = os.listdir(DATA_DIR)
datasets

['GSE156625', 'GSE166635', 'GSE125449', 'GSE149614', 'ICB_dataset']

# GSE149614


In [12]:
import os

dataset_path = f"{DATA_DIR}/GSE149614"
os.listdir(dataset_path)

['GSE149614_HCC.scRNAseq.S71915.count.txt',
 'GSE149614_HCC.metadata.updated.txt']

In [14]:
# Construir ANNdata manualmente


counts_path = f"{DATA_DIR}/GSE149614/GSE149614_HCC.scRNAseq.S71915.count.txt"

preview = pd.read_csv(counts_path, sep="\t", nrows=5)
preview

,HCC01T_AAACCTGAGGGCATGT,HCC01T_AAACCTGAGTCGCCGT,HCC01T_AAACCTGCATTACCTT,HCC01T_AAACCTGGTCACACGC,HCC01T_AAACCTGTCCAGTATG,HCC01T_AAACGGGTCGAGCCCA,HCC01T_AAACGGGTCGCTTGTC,HCC01T_AAACGGGTCTGACCTC,HCC01T_AAAGATGAGACAGACC,HCC01T_AAAGATGAGCTAACAA,...,HCC06T_TTTGGTTTCTCGTTTA,HCC06T_TTTGTCAAGCGTGTCC,HCC06T_TTTGTCAAGTCCTCCT,HCC06T_TTTGTCACAAATCCGT,HCC06T_TTTGTCACAGATCTGT,HCC06T_TTTGTCACAGTATGCT,HCC06T_TTTGTCAGTCCAAGTT,HCC06T_TTTGTCAGTTTGCATG,HCC06T_TTTGTCATCCTGTACC,HCC06T_TTTGTCATCGACAGCC
RP11-34P13.7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
FO538757.2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AP006222.2,1,1,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
RP4-669L17.10,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
RP5-857K21.4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
# Cargar metadata

metadata_path = f"{DATA_DIR}/GSE149614/GSE149614_HCC.metadata.updated.txt"
metadata = pd.read_csv(metadata_path, sep="\t")

print(metadata.shape)
metadata.head()

# IDs celulas count

with open(counts_path, "r") as f:
    header = f.readline().rstrip("\n").split("\t")

cell_ids_counts = header[1:]
print("Células en counts:", len(cell_ids_counts))
print(cell_ids_counts[:5])

# IDs del metadata

cell_ids_meta = metadata["Cell"].tolist()
print("Células en metadata:", len(cell_ids_meta))
print(cell_ids_meta[:5])

# Comprobar si coinciden

set_counts = set(cell_ids_counts)
set_meta = set(cell_ids_meta)

print("En counts pero no en metadata:", len(set_counts - set_meta))
print("En metadata pero no en counts:", len(set_meta - set_counts))

#  Mismo orden?

print("¿Mismo orden exacto?:", cell_ids_counts == cell_ids_meta)



(71915, 8)
Células en counts: 71914
['HCC01T_AAACCTGAGTCGCCGT', 'HCC01T_AAACCTGCATTACCTT', 'HCC01T_AAACCTGGTCACACGC', 'HCC01T_AAACCTGTCCAGTATG', 'HCC01T_AAACGGGTCGAGCCCA']
Células en metadata: 71915
['HCC01T_AAACCTGAGGGCATGT', 'HCC01T_AAACCTGAGTCGCCGT', 'HCC01T_AAACCTGCATTACCTT', 'HCC01T_AAACCTGGTCACACGC', 'HCC01T_AAACCTGTCCAGTATG']
En counts pero no en metadata: 0
En metadata pero no en counts: 1
¿Mismo orden exacto?: False


Falta una célula del metadata en counts

In [16]:
# Identificar que célula sobra

missing_in_counts = list(set_meta - set_counts)
print(missing_in_counts)



['HCC01T_AAACCTGAGGGCATGT']


In [17]:
# Quitar del metadata la célula que no está en counts
metadata_filt = metadata[metadata["Cell"].isin(cell_ids_counts)].copy()

print(metadata_filt.shape) 

# Poner Cell como índice
metadata_filt = metadata_filt.set_index("Cell")

# Reordenar para que siga exactamente el orden del counts
metadata_filt = metadata_filt.loc[cell_ids_counts]

# Comprobar
print(metadata_filt.index.tolist() == cell_ids_counts)  
print(metadata_filt.head())

(71914, 8)
True
                         sample  res.3   site patient stage virus celltype
Cell                                                                      
HCC01T_AAACCTGAGTCGCCGT  HCC01T     16  Tumor   HCC01     I   HBV  Myeloid
HCC01T_AAACCTGCATTACCTT  HCC01T     25  Tumor   HCC01     I   HBV     T/NK
HCC01T_AAACCTGGTCACACGC  HCC01T      2  Tumor   HCC01     I   HBV     T/NK
HCC01T_AAACCTGTCCAGTATG  HCC01T      2  Tumor   HCC01     I   HBV     T/NK
HCC01T_AAACGGGTCGAGCCCA  HCC01T     38  Tumor   HCC01     I   HBV  Myeloid


In [18]:
test_chunk = next(pd.read_csv(
    counts_path,
    sep="\t",
    index_col=0,
    chunksize=200
))

test_chunk = test_chunk[cell_ids_counts]

In [19]:
# Convertimos el chunk a una matriz numpy para poder operar de forma eficiente
arr = test_chunk.to_numpy()

# Obtenemos las posiciones (fila, columna) donde los valores son distintos de 0
rows, cols = np.nonzero(arr)

# Extraemos los valores distintos de 0 (expresión génica real)
vals = arr[rows, cols]

# Calculamos el número total de valores posibles en la matriz
total_values = arr.shape[0] * arr.shape[1]

# Calculamos cuántos de esos valores son distintos de 0
nonzero_values = len(vals)

# Calculamos la proporción de valores no nulos (nivel de dispersión de la matriz)
print("Justificación de uso de estructuras sparse para optimizar memoria y procesamiento")
print("Total valores:", total_values)
print("No ceros:", nonzero_values)
print("Proporción no cero:", nonzero_values / total_values)

Justificación de uso de estructuras sparse para optimizar memoria y procesamiento
Total valores: 14382800
No ceros: 1303169
Proporción no cero: 0.09060607114052896


La proporción de valores no nulos en este fragmento de la matriz es ~9 %,
lo que indica que la matriz de expresión es altamente dispersa.
Este patrón es esperado en datos de scRNA-seq, donde muchos genes no se
expresan en muchas células. Por ello, el uso de estructuras sparse es
adecuado para optimizar memoria y procesamiento.

In [21]:
gene_names = []
data_list = []
row_list = []
col_list = []

gene_offset = 0
chunksize = 1000  # más grande = más rápido

for chunk in pd.read_csv(
    counts_path,
    sep="\t",
    index_col=0,
    chunksize=chunksize
):
    chunk = chunk[cell_ids_counts]

    gene_names.extend(chunk.index.tolist())

    arr = chunk.values

    rows, cols = np.nonzero(arr)
    vals = arr[rows, cols]

    row_list.append(rows + gene_offset)
    col_list.append(cols)
    data_list.append(vals)

    gene_offset += arr.shape[0]

    if gene_offset % 5000 == 0:
        print(f"Procesados {gene_offset} genes")

Procesados 5000 genes
Procesados 10000 genes
Procesados 15000 genes
Procesados 20000 genes
Procesados 25000 genes


Debido al gran tamaño de los datos de scRNA-seq y a que la mayoría de los valores son ceros, no es eficiente cargar todo el archivo en memoria de una sola vez. Por ello, se leyó el archivo por partes (chunks), procesando un número reducido de genes en cada iteración. Además, solo se almacenaron los valores distintos de cero junto con sus posiciones en la matriz, lo que permitió construir una matriz dispersa (sparse) más eficiente en términos de memoria y procesamiento.

In [22]:
# Unir todo
rows = np.concatenate(row_list).astype(np.int32)
cols = np.concatenate(col_list).astype(np.int32)
data = np.concatenate(data_list).astype(np.float32)

del row_list
del col_list
del data_list

In [23]:
print(data.nbytes / 1e6, "MB")

586.640108 MB


In [26]:
from scipy.sparse import coo_matrix

sparse_matrix = coo_matrix(
    (data, (rows, cols)),
    shape=(len(gene_names), len(cell_ids_counts))
)

del rows
del cols
del data

In [27]:
sparse_matrix = sparse_matrix.tocsr()

In [29]:
# Crear el objeto AnnData y comprobar que todo está correcto
import anndata as ad

adata = ad.AnnData(
    X=sparse_matrix.T,
    obs=metadata_filt,
    var=pd.DataFrame(index=gene_names)
)

print(adata)
print(adata.shape)
adata.obs.head()
adata.var.head()

AnnData object with n_obs × n_vars = 71914 × 25712
    obs: 'sample', 'res.3', 'site', 'patient', 'stage', 'virus', 'celltype'
(71914, 25712)


""
RP11-34P13.7
FO538757.2
AP006222.2
RP4-669L17.10
RP5-857K21.4


In [30]:
# GUARDARLO!

adata.write(f"{DATA_DIR}/adata_GSE149614_raw.h5ad")

In [3]:
import os

print("Directorio actual:", os.getcwd())
print("DATA_DIR:", DATA_DIR)
print("Ruta completa:", os.path.join(DATA_DIR, "adata_GSE149614_raw.h5ad"))
print("Existe?", os.path.exists(os.path.join(DATA_DIR, "adata_GSE149614_raw.h5ad")))

Directorio actual: /beegfs/home/iruizdealda/HCC_singlecell_project/notebooks
DATA_DIR: ../data
Ruta completa: ../data/adata_GSE149614_raw.h5ad
Existe? True
